In [2]:
import time
import pandas as pd
from spam_shield.config import PROCESSED_DATASET_FILE
from spam_shield.preprocessing import preprocess_texts

df = pd.read_csv(PROCESSED_DATASET_FILE)

start = time.time()
df["processed_text"] = preprocess_texts(df["text_content"].tolist())
print(f"Processed {len(df):,} messages in {time.time() - start:.0f} seconds")

df[["text_content", "processed_text", "label"]].sample(5, random_state=42)

Processed 38,656 messages in 255 seconds


,text_content,processed_text,label
24812,"""\nPOV-ridden article? Yeah, sure! BTW, I am g...",pov ridden article yeah sure btw confuse prune...,ham
4921,"""Hi missed your Call and my mumHas beendroppin...",hi miss mumhas beendroppe red wine theplace ad...,ham
4884,Did you see that film:),film,ham
31495,page. Untill then don't waste my time by posti...,page untill not waste time post crap thing,toxic
14324,otc - guide on great investment homeland secur...,otc guide great investment homeland security i...,phishing


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from spam_shield import config

train = pd.read_csv(config.TRAIN_FILE)
train["processed_text"] = train["processed_text"].fillna("")

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # single words AND pairs of words ("free prize")
    min_df=3,               # ignore words that appear in fewer than 3 messages
    max_df=0.9,             # ignore words that appear in more than 90% of messages
    max_features=50_000,    # keep at most the 50,000 most frequent words
    sublinear_tf=True,      # a word repeated 100 times is not 100x more important
)
X_train = vectorizer.fit_transform(train["processed_text"])
print("Shape:", X_train.shape)

words = vectorizer.get_feature_names_out()
for label in config.LABELS:
    rows = (train["label"] == label).to_numpy()
    mean_scores = np.asarray(X_train[rows].mean(axis=0)).ravel()
    top = mean_scores.argsort()[::-1][:15]
    print(f"\n{label.upper()}: {', '.join(words[top])}")

Shape: (30924, 50000)

HAM: not, url, thank, article, talk, know, use, no, page, like, think, time, list, email, work

SPAM: txt, free, mobile, ur, text, win, reply, prize, 150p, stop, claim, min, urgent, nokia, tone

PHISHING: url, click, email, no, free, not, remove, offer, receive, com, money, online, good, price, time

TOXIC: fuck, not, shit, suck, fucking, like, page, block, edit, ass, bitch, stupid, idiot, article, stop


In [3]:
from spam_shield.features import extract_metadata

meta = extract_metadata(train["text_content"])
meta.groupby(train["label"]).mean().round(3).T

label,ham,phishing,spam,toxic
char_count,1005.786,1668.393,137.426,303.590
word_count,188.319,317.214,23.706,52.840
url_count,0.444,0.726,0.128,0.028
email_count,0.177,0.154,0.010,0.001
exclamation_count,0.472,3.457,0.691,3.158
currency_count,0.363,2.702,0.422,0.010
uppercase_ratio,0.033,0.018,0.111,0.107
digit_ratio,0.021,0.022,0.116,0.012
